# 獲得関数の初期値生成：マルチスタート最適化とqKG

獲得関数の最大化で使う**初期候補（initial conditions）**の役割を、通常の `qLogExpectedImprovement` と one-shot `qKnowledgeGradient` で比較します。

初期値生成 → `robotorchan.optim.optimize_acqf` → 候補評価 → GP再学習 → 次の提案まで実行します。BoTorchとrobotorchanの既存公開APIを利用します。


## 1. Imports and reproducibility

In [ ]:
import torch
from botorch.acquisition.knowledge_gradient import qKnowledgeGradient
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.optim.initializers import (
    gen_batch_initial_conditions,
    gen_one_shot_kg_initial_conditions,
)
from botorch.sampling.normal import SobolQMCNormalSampler

from robotorchan.models import SingleTaskGP
from robotorchan.optim import optimize_acqf

torch.set_default_dtype(torch.double)
torch.manual_seed(0)

### なぜ獲得関数にも最適化が必要なのか

GPを学習しただけでは次の実験条件は決まりません。獲得関数を最大化することで、**モデルの予測と不確実性に基づく次の候補**を求めます。

獲得関数の形が複雑な場合、1つの初期値からの局所最適化では良い解に到達できないことがあります。そのため複数の開始点を生成し、最も良い結果を選ぶ**マルチスタート最適化**を利用します。

## 2. Fit the surrogate and construct qLogEI

In [ ]:
def response(X: torch.Tensor) -> torch.Tensor:
    return torch.sin(2 * torch.pi * X) + 0.2 * X


train_X = torch.linspace(0.05, 0.95, 9).unsqueeze(-1)
train_Y = response(train_X)
model = SingleTaskGP(train_X, train_Y)
model.fit()
model.eval()

bounds = torch.tensor([[0.0], [1.0]])
acquisition = qLogExpectedImprovement(
    model=model,
    best_f=train_Y.max(),
    sampler=SobolQMCNormalSampler(torch.Size([64]), seed=101),
)

## 3. Standard q-batch initial conditions

通常のexplicit initial-condition shapeは`num_restarts x q x d`です。

In [ ]:
num_restarts = 4
q = 2
initial_conditions = gen_batch_initial_conditions(
    acq_function=acquisition,
    bounds=bounds,
    q=q,
    num_restarts=num_restarts,
    raw_samples=64,
)

assert initial_conditions.shape == torch.Size([num_restarts, q, 1])
print("initial-condition shape:", initial_conditions.shape)
print(initial_conditions)

### `raw_samples` と `num_restarts` は別の数

`raw_samples=64` は初期値を選ぶために評価する生の候補数、`num_restarts=4` は局所最適化を始める点の数です。

通常の `q=2` では1つの開始点が**2候補の組**であり、初期値テンソルは `[4,2,1]`（4開始点 × 2候補 × 1入力次元）になります。64回の局所最適化を実行するという意味ではありません。

## 4. Explicit starts are consumed by the optimizer

`batch_initial_conditions`を渡す場合、そのrestart setがlocal optimizationの開始点になります。

In [ ]:
candidate, value = optimize_acqf(
    acq_function=acquisition,
    bounds=bounds,
    q=q,
    optimizer="botorch",
    num_restarts=num_restarts,
    raw_samples=None,
    batch_initial_conditions=initial_conditions,
)

assert candidate.shape == torch.Size([q, 1])
print("candidate:", candidate.squeeze(-1))
print("acquisition value:", value)

### 初期値を明示的に渡す場合

`gen_batch_initial_conditions()` が返す開始点を `batch_initial_conditions` に渡します。初期値生成を自動で行う場合と異なり、どこから局所最適化を始めるかを確認できます。

**初期値は最終候補ではありません**。獲得関数の局所最適化後に返された `candidate` が、実験で評価する入力です。

## 5. Fixed features participate in initialization

fixed coordinateはoptimizationだけでなくinitial-condition generationでも保持される必要があります。

In [ ]:
train_X_2d = torch.rand(12, 2)
train_Y_2d = (
    torch.sin(2 * torch.pi * train_X_2d[:, :1])
    + 0.25 * torch.cos(2 * torch.pi * train_X_2d[:, 1:2])
)
model_2d = SingleTaskGP(train_X_2d, train_Y_2d)
model_2d.fit()
model_2d.eval()
acq_2d = qLogExpectedImprovement(model=model_2d, best_f=train_Y_2d.max())
bounds_2d = torch.tensor([[0.0, 0.0], [1.0, 1.0]])

fixed_initial = gen_batch_initial_conditions(
    acq_function=acq_2d,
    bounds=bounds_2d,
    q=1,
    num_restarts=4,
    raw_samples=64,
    fixed_features={1: 0.4},
)

torch.testing.assert_close(fixed_initial[..., 1], torch.full((4, 1), 0.4))
print("fixed-feature starts:\n", fixed_initial)

### 固定変数の意味

2次元入力のうち第2成分を `0.4` に固定する例です。固定変数は初期値生成だけでなく、実際の候補最適化でも守る必要があります。

ここでは初期値生成時の固定を確認します。実務で候補まで固定したい場合は、最適化側にも同じ `fixed_features` を渡します。

## 6. Pending points do not add an initialization axis

`X_pending`はacquisition contextです。通常initial conditionsはpending rowsの数に関係なく`[restart, q, d]`のままです。

### `X_pending` と最適化する候補は別

`X_pending` は既に実験を開始したものの結果がまだ返っていない入力です。新たに提案する候補の数 `q` には含まれません。

したがってpending点が2つあっても、今回新たに2候補を提案する初期値の形状は `[num_restarts,2,d]` のままです。

In [ ]:
X_pending = torch.tensor([[0.30], [0.70]])
pending_acq = qLogExpectedImprovement(
    model=model,
    best_f=train_Y.max(),
    sampler=SobolQMCNormalSampler(torch.Size([64]), seed=202),
    X_pending=X_pending,
)
pending_initial = gen_batch_initial_conditions(
    acq_function=pending_acq,
    bounds=bounds,
    q=2,
    num_restarts=4,
    raw_samples=64,
)

assert pending_initial.shape == torch.Size([4, 2, 1])
print("pending-aware initial-condition shape:", pending_initial.shape)

## 7. Knowledge Gradient is a one-shot exception

qKGはreal candidateにfantasy/auxiliary decision rowsを加えたaugmented problemを最適化します。したがってordinary `[restart, q, d]` startsをそのまま使いません。

### qKGでは補助変数も同時に最適化する

`qKnowledgeGradient` は将来の観測結果を仮想的に考慮する**lookahead獲得関数**です。one-shot形式では、実際に測定する入力のほかにfantasyに対応する補助的な最適化変数を使います。

この例は実候補1点と補助変数4行の計5行を内部最適化します。ただし、**実験で測定するのは公開候補1点だけ**です。

In [ ]:
kg = qKnowledgeGradient(
    model=model,
    num_fantasies=4,
)
public_q = 1
augmented_q = kg.get_augmented_q_batch_size(q=public_q)
kg_initial = gen_one_shot_kg_initial_conditions(
    acq_function=kg,
    bounds=bounds,
    q=public_q,
    num_restarts=3,
    raw_samples=32,
)

assert augmented_q == 5
assert kg_initial.shape == torch.Size([3, augmented_q, 1])
print("public q:", public_q)
print("augmented q:", augmented_q)
print("qKG initial-condition shape:", kg_initial.shape)

## 8. qKG optimization returns only public candidates

optimizer内部ではaugmented batchを扱いますが、one-shot acquisitionのcandidate extractionにより返されるreal candidateはrequested public qです。

### qKGで実験するのは公開候補だけ

`get_augmented_q_batch_size()` が返すサイズは最適化内部の変数数です。`optimize_acqf()` が返す `kg_candidate` の行数は、指定した実候補数 `public_q` になります。

補助的なfantasy変数をそのまま測定装置へ渡してはいけません。

In [ ]:
kg_candidate, kg_value = optimize_acqf(
    acq_function=kg,
    bounds=bounds,
    q=public_q,
    optimizer="botorch",
    num_restarts=3,
    raw_samples=32,
)

assert kg_candidate.shape == torch.Size([public_q, 1])
print("qKG public candidate:", kg_candidate)
print("qKG acquisition value:", kg_value)

### qLogEIの2候補を観測してGPを再学習する

最初に提案した `candidate` は2点です。人工関数 `response` を測定器の代わりにして実際の応答を取得し、観測データに追加します。

次に、更新したGPで獲得関数を作り直し、新しい初期値から次の2候補を提案します。これにより初期化がBOループのどこに位置するか確認できます。

In [ ]:
observed_Y = response(candidate.detach())
updated_X = torch.cat([train_X, candidate.detach()], dim=0)
updated_Y = torch.cat([train_Y, observed_Y], dim=0)
updated_model = SingleTaskGP(updated_X, updated_Y)
updated_model.fit()
updated_model.eval()
updated_acquisition = qLogExpectedImprovement(
    model=updated_model,
    best_f=updated_Y.max(),
    sampler=SobolQMCNormalSampler(torch.Size([64]), seed=102),
)
next_initial_conditions = gen_batch_initial_conditions(
    acq_function=updated_acquisition,
    bounds=bounds,
    q=2,
    num_restarts=num_restarts,
    raw_samples=64,
)
next_candidate, next_value = optimize_acqf(
    acq_function=updated_acquisition,
    bounds=bounds,
    q=2,
    optimizer="botorch",
    num_restarts=num_restarts,
    raw_samples=None,
    batch_initial_conditions=next_initial_conditions,
)
print("Updated observation count:", updated_X.shape[0])
print("Next q=2 candidates:", next_candidate.squeeze(-1))
print("Next acquisition value:", next_value)
assert updated_X.shape[0] == train_X.shape[0] + 2
assert next_initial_conditions.shape == (num_restarts, 2, 1)
assert next_candidate.shape == (2, 1)

### 観測前後の予測を確認する

初期値は獲得関数の最適化に関わる設定で、GPの学習データそのものではありません。GPの事後分布が変わるのは、**候補を観測してデータを追加し、再学習したとき**です。

In [ ]:
import matplotlib.pyplot as plt

plot_X = torch.linspace(0.0, 1.0, 201).unsqueeze(-1)
with torch.no_grad():
    before_posterior = model.posterior(plot_X)
    after_posterior = updated_model.posterior(plot_X)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(plot_X.squeeze(-1), response(plot_X).squeeze(-1), label="True response")
for post, label in ((before_posterior, "Before"), (after_posterior, "After")):
    center = post.mean.squeeze(-1)
    std = post.variance.clamp_min(0).sqrt().squeeze(-1)
    ax.plot(plot_X.squeeze(-1), center, label=f"{label} mean")
    ax.fill_between(plot_X.squeeze(-1), center - 2 * std, center + 2 * std, alpha=0.12)
ax.scatter(candidate.squeeze(-1), observed_Y.squeeze(-1), marker="x", s=70, label="New observations")
ax.set(xlabel="x", ylabel="response", title="Posterior update after q=2 observations")
ax.legend()
plt.show()

## 9. Initialization axes must remain distinct

| Quantity | Initialization meaning |
| --- | --- |
| `num_restarts` | independent local-search starts |
| ordinary `q` | real joint candidate rows |
| `d` | current search-coordinate dimension |
| `X_pending` rows | acquisition context; not an IC axis |
| fantasy-model batch | model batch; not an IC axis |
| robust perturbation scenarios | uncertainty scenarios; not an IC axis |
| one-shot auxiliary rows | optimization variables; therefore augmented q |

## 10. Geometry-specific initialization

通常box search以外ではinitialization coordinate自体が変わります。

- REMBO / HeSBO: embedded coordinates
- ALEBO: feasible polytope sampling
- TuRBO: current trust-region bounds
- Hybrid: global-stage candidate becomes local explicit start
- Mixed one-shot: augmented rowsごとのcategorical assignment semantics

これらをpublic input-spaceのstandard box initializerへ機械的に押し込みません。

## 11. Constraint boundary

linear constraintsではBoTorchがfeasible startsを生成できます。nonlinear constraintsではfeasible `batch_initial_conditions`または適切な`ic_generator`が必要です。inter-point constraintsはjoint q relationなので、独立q=1 initializationへ分解できません。

## 12. Related documentation

- `docs/theory/optimization/03_initialization.md`
- `docs-internal/verification/optimization/initialization-compatibility.md`
- `docs/optimization/initialization.md`
- `39_acquisition_optimization.ipynb`
- `40_batch_async_fantasization.ipynb`

Initializationはoptimizerの付属パラメータではなく、search geometryとq semanticsを保つための契約です。

## 初期化から提案された候補の評価

qKGの補助fantasy変数は評価対象ではありません。optimizerが返す公開候補のみを目的関数で評価します。


In [ ]:
kg_observed_Y = response(kg_candidate.detach())
assert kg_observed_Y.shape[0] == public_q
print("qKG candidate evaluation:", kg_observed_Y)


## 13. 初期化に関する公開APIの整理

| 役割 | 既存API | 注意点 |
|---|---|---|
| 通常のq-batch初期値 | `gen_batch_initial_conditions` | `[num_restarts,q,d]` |
| one-shot qKG初期値 | `gen_one_shot_kg_initial_conditions` | 補助行を含むaugmented q |
| 候補最適化 | `robotorchan.optim.optimize_acqf` | 初期値を明示的に指定可能 |
| 観測後の再学習 | `SingleTaskGP.fit()` | 観測データを追加してから実行 |
| qKG実候補の取得 | `optimize_acqf` の戻り値 | fantasy補助行は実験対象外 |

**重要：** `raw_samples`、`num_restarts`、`q` は異なる意味を持ちます。初期化は探索を助けますが、獲得関数の選択やGPの事後分布を直接変更するものではありません。